In [2]:
#Imports
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [3]:
BASE_DIR = Path("../")
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"

In [4]:
TRAIN_FILES = [
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
]

In [5]:
TEST_FILES = [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]

In [6]:
#Checking test files 
print("Training files:")
for f in TRAIN_FILES:
    print("-", f)

print("\nTest files:")
for f in TEST_FILES:
    print("-", f)

Training files:
- Tuesday-WorkingHours.pcap_ISCX.csv
- Wednesday-workingHours.pcap_ISCX.csv
- Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
- Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
- Friday-WorkingHours-Morning.pcap_ISCX.csv

Test files:
- Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


In [10]:
def standardize_csv(file_path):
    df = pd.read_csv(file_path)
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

In [11]:
#Load and combine all training CSV 
train_dfs = []

#Standardize, append and concatenate 
for file_name in TRAIN_FILES:
    file_path = RAW_DATA_DIR / file_name
    df = standardize_csv(file_path)
    #append standardized dataframe to train dataframe list
    train_dfs.append(df)
    print(f"Loaded {file_name} with shape {df.shape}")
    
train_df = pd.concat(train_dfs, ignore_index= True)
print("\nCombined training data shape:", train_df.shape)

Loaded Tuesday-WorkingHours.pcap_ISCX.csv with shape (445909, 79)
Loaded Wednesday-workingHours.pcap_ISCX.csv with shape (692703, 79)
Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv with shape (170366, 79)
Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv with shape (286467, 79)
Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv with shape (191033, 79)

Combined training data shape: (1786478, 79)


In [12]:
#Load and combine all test CSV
test_dfs = []

#append standardize dataframe to test dataframe list 
for file_name in TEST_FILES:
    file_path = RAW_DATA_DIR / file_name
    df = standardize_csv(file_path)
    test_dfs.append(df)
    print(f"Loaded {file_name} with shape {df.shape}")

test_df = pd.concat(test_dfs, ignore_index=True)
print("\nCombined test data shape:", test_df.shape)

Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv with shape (225745, 79)

Combined test data shape: (225745, 79)


In [13]:
print("Train columns == Test columns:", list(train_df.columns) == list(test_df.columns))
print("Number of train columns:", len(train_df.columns))
print("Number of test columns:", len(test_df.columns))

Train columns == Test columns: True
Number of train columns: 79
Number of test columns: 79


In [15]:
#EDA
train_df.head
test_df.head

<bound method NDFrame.head of         destination_port  flow_duration  total_fwd_packets  \
0                  54865              3                  2   
1                  55054            109                  1   
2                  55055             52                  1   
3                  46236             34                  1   
4                  54863              3                  2   
...                  ...            ...                ...   
225740             61374             61                  1   
225741             61378             72                  1   
225742             61375             75                  1   
225743             61323             48                  2   
225744             61326             68                  1   

        total_backward_packets  total_length_of_fwd_packets  \
0                            0                           12   
1                            1                            6   
2                            1      

In [16]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (1786478, 79)
Test shape: (225745, 79)


In [17]:
#Label distrubution 
print("Train label distribution:")
print(train_df["label"].value_counts())

print("\nTest label distribution:")
print(test_df["label"].value_counts())

Train label distribution:
label
BENIGN                        1356895
DoS Hulk                       231073
PortScan                       158930
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Test label distribution:
label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


In [18]:
print("Train label proportions:")
print(train_df["label"].value_counts(normalize=True))

print("\nTest label proportions:")
print(test_df["label"].value_counts(normalize=True))

Train label proportions:
label
BENIGN                        0.759536
DoS Hulk                      0.129346
PortScan                      0.088963
DoS GoldenEye                 0.005762
FTP-Patator                   0.004443
SSH-Patator                   0.003301
DoS slowloris                 0.003244
DoS Slowhttptest              0.003078
Bot                           0.001100
Web Attack � Brute Force      0.000844
Web Attack � XSS              0.000365
Web Attack � Sql Injection    0.000012
Heartbleed                    0.000006
Name: proportion, dtype: float64

Test label proportions:
label
DDoS      0.567131
BENIGN    0.432869
Name: proportion, dtype: float64


In [19]:
print("Missing values in train:")
print(train_df.isnull().sum().sort_values(ascending=False).head(10))

print("\nMissing values in test:")
print(test_df.isnull().sum().sort_values(ascending=False).head(10))

Missing values in train:
flow_bytes/s                   1272
flow_duration                     0
destination_port                  0
total_backward_packets            0
total_length_of_fwd_packets       0
total_length_of_bwd_packets       0
total_fwd_packets                 0
fwd_packet_length_max             0
fwd_packet_length_min             0
fwd_packet_length_std             0
dtype: int64

Missing values in test:
flow_bytes/s                   4
flow_duration                  0
destination_port               0
total_backward_packets         0
total_length_of_fwd_packets    0
total_length_of_bwd_packets    0
total_fwd_packets              0
fwd_packet_length_max          0
fwd_packet_length_min          0
fwd_packet_length_std          0
dtype: int64


In [20]:
train_inf_count = np.isinf(train_df.select_dtypes(include=[np.number])).sum().sum()
test_inf_count = np.isinf(test_df.select_dtypes(include=[np.number])).sum().sum()

print("Infinite values in train:", train_inf_count)
print("Infinite values in test:", test_inf_count)

Infinite values in train: 3106
Infinite values in test: 64


In [21]:
#Replace inf with NaN
train_df.replace([np.inf, -np.inf], np.nan, inplace=True)
test_df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [22]:
train_df = train_df.dropna()
test_df = test_df.dropna()

print("Train shape after dropna:", train_df.shape)
print("Test shape after dropna:", test_df.shape)

Train shape after dropna: (1784289, 79)
Test shape after dropna: (225711, 79)


In [25]:
train_df.loc[:, "label"] = train_df["label"].apply(lambda x: 0 if x == "BENIGN" else 1)
test_df.loc[:, "label"] = test_df["label"].apply(lambda x: 0 if x == "BENIGN" else 1)

In [26]:
print("Binary train labels:")
print(train_df["label"].value_counts())

print("\nBinary test labels:")
print(test_df["label"].value_counts())

Binary train labels:
label
1    1784289
Name: count, dtype: int64

Binary test labels:
label
1    225711
Name: count, dtype: int64


In [27]:
X_train_full = train_df.drop("label", axis=1)
y_train_full = train_df["label"]

X_test = test_df.drop("label", axis=1)
y_test = test_df["label"]

print("X_train_full:", X_train_full.shape)
print("y_train_full:", y_train_full.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train_full: (1784289, 78)
y_train_full: (1784289,)
X_test: (225711, 78)
y_test: (225711,)


In [29]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (1427431, 78)
X_val: (356858, 78)
y_train: (1427431,)
y_val: (356858,)


In [30]:
summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(X_train), len(X_val), len(X_test)],
    "columns": [X_train.shape[1], X_val.shape[1], X_test.shape[1]],
    "attack_ratio": [y_train.mean(), y_val.mean(), y_test.mean()]
})

summary

,split,rows,columns,attack_ratio
0,train,1427431,78,1.0
1,val,356858,78,1.0
2,test,225711,78,1.0


In [32]:
os.makedirs(PROCESSED_DATA_DIR / "train", exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR / "val", exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR / "test", exist_ok=True)

In [33]:
X_train.to_csv(PROCESSED_DATA_DIR / "train" / "X_train.csv", index=False)
y_train.to_csv(PROCESSED_DATA_DIR / "train" / "y_train.csv", index=False)

X_val.to_csv(PROCESSED_DATA_DIR / "val" / "X_val.csv", index=False)
y_val.to_csv(PROCESSED_DATA_DIR / "val" / "y_val.csv", index=False)

X_test.to_csv(PROCESSED_DATA_DIR / "test" / "X_test.csv", index=False)
y_test.to_csv(PROCESSED_DATA_DIR / "test" / "y_test.csv", index=False)

print("Processed splits saved successfully.")

Processed splits saved successfully.
